# Stage 4: Final Evidence on Untouched Test Set & FAISS Comparison

**Milestone A2**: Knowledge Base Verification, FAISS Index Benchmarks, & Human Citations.

### Protocol:
1. Consumes `candidate-lock.json` emitted by `stage4-dev-selection.ipynb`.
2. Evaluates the locked winning stack **once** on the untouched **Final Test Split** (20 queries: 15 single + 5 multi).
3. Benchmarks FAISS `IndexFlatIP`, `IndexHNSWFlat`, and `IndexIVFFlat` using normalized inner product.
4. Records 1 successful retrieval example and 1 worst failure example with full provenance.
5. Persists the production knowledge base and emits `stage4-final-evidence.zip`.

### 1. Environment & Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if Path("/kaggle/working").is_dir():
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    INPUT_ROOT = Path("/kaggle/input")
    OUT_DIR = Path("/kaggle/working/final_output")
    LOCK_PATH = Path("/kaggle/working/dev_output/candidate-lock.json")
    if not LOCK_PATH.is_file():
        found = list(INPUT_ROOT.rglob("candidate-lock.json"))
        if found:
            LOCK_PATH = found[0]
else:
    INPUT_ROOT = Path("extras/indexing-benchmarks")
    OUT_DIR = Path("results/final_output")
    LOCK_PATH = Path("results/dev_output/candidate-lock.json")

OUT_DIR.mkdir(parents=True, exist_ok=True)
KB_DIR = OUT_DIR / "production_kb"
KB_DIR.mkdir(parents=True, exist_ok=True)

### 2. Load Modules & Candidate Lock

In [ ]:
for p in [Path("code"), Path("../code"), Path("extras/indexing-benchmarks/code"), *INPUT_ROOT.rglob("code")]:
    if p.is_dir() and (p / "corpus.py").is_file():
        if str(p.resolve()) not in sys.path:
            sys.path.insert(0, str(p.resolve()))
        print(f"Loaded benchmark package from: {p.resolve()}")
        break

import json
from corpus import load_canonical_corpus
from queries import load_retrieval_queries
from chunking import build_chunk_suites
from models import EmbeddingModelAdapter
from evaluation import evaluate_retrieval_suite
from faiss_benchmark import benchmark_faiss_architectures

if not LOCK_PATH.is_file():
    raise FileNotFoundError(f"Missing candidate lock file: {LOCK_PATH}. Run stage4-dev-selection.ipynb first!")

lock_data = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
winning_model_id = lock_data["winning_model_id"]
winning_model_path = lock_data["resolved_model_path"]
winning_strategy = lock_data["winning_chunk_strategy"]

print("\n--- Consumed Candidate Lock ---")
print(f"Winning Model:    {winning_model_id} ({lock_data['dimension']}-d)")
print(f"Winning Chunking: {winning_strategy}")
print(f"Dev Recall@5:     {lock_data['dev_recall@5']:.4f} (95% CI: {lock_data['dev_recall@5_ci_95']})")

### 3. Load Untouched Final Test Set (20 Queries)

In [ ]:
pages = load_canonical_corpus(INPUT_ROOT)
all_queries = load_retrieval_queries(INPUT_ROOT)
test_queries = [q for q in all_queries if q.split == "test"]

print("\n--- Final Test Split ---")
print(f"Total Pages: {len(pages)}")
print(f"Untouched Test Queries: {len(test_queries)} (15 single-page + 5 multi-page)")
assert len(test_queries) == 20, f"Expected 20 test queries, got {len(test_queries)}"

# Build winning chunks
chunk_suites = build_chunk_suites(pages)
winning_chunks = chunk_suites[winning_strategy]
print(f"Winning Chunk Suite: {len(winning_chunks)} chunks")

### 4. One-Time Untouched Final Test Evaluation

In [ ]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

adapter = EmbeddingModelAdapter(winning_model_path, canonical_id=winning_model_id, device=device)
is_pc = (winning_strategy == "parent_child_128_512")

final_metrics, test_logs = evaluate_retrieval_suite(
    adapter, winning_chunks, test_queries, top_k=10, is_parent_child=is_pc
)

print("\n" + "=" * 80)
print("FINAL TEST EVALUATION RESULTS (Untouched Split)")
print("=" * 80)
print(f"Single-Page Recall@1:         {final_metrics['single_page_recall@1']:.4f}")
print(f"Single-Page Recall@5:         {final_metrics['single_page_recall@5']:.4f} (95% CI: {final_metrics['recall@5_ci_95']})")
print(f"Single-Page Recall@10:        {final_metrics['single_page_recall@10']:.4f}")
print(f"Single-Page MRR@10:           {final_metrics['single_page_mrr@10']:.4f}")
print(f"Exact Span Containment@5:     {final_metrics['single_page_span_containment@5']:.4f}")
print(f"Multi-Page Page Coverage@10:  {final_metrics['multi_page_coverage@10']:.4f}")
print(f"Multi-Page All-Found@10:      {final_metrics['multi_page_all_found@10']:.4f}")
print("=" * 80)

### 5. FAISS Vector Search Architecture Comparison

In [ ]:
import faiss

print("Benchmarking FAISS Vector Index Architectures on final normalized vectors...")
final_doc_embs = adapter.encode_documents([c.text for c in winning_chunks])
final_q_embs = adapter.encode_queries([q.question for q in test_queries])

exact_idx = faiss.IndexFlatIP(final_doc_embs.shape[1])
exact_idx.add(final_doc_embs)
_, gt_top10 = exact_idx.search(final_q_embs, 10)

faiss_summary = benchmark_faiss_architectures(final_doc_embs, final_q_embs, gt_top10, num_iterations=10)

print("\n" + "-" * 90)
print(f"{'Index Architecture':<20} {'Build Time (s)':<16} {'P50 Latency (ms)':<18} {'P95 Latency (ms)':<18} {'Top-10 Agreement':<16}")
print("-" * 90)
for idx_name, info in faiss_summary.items():
    print(f"{idx_name:<20} {info['build_time_s']:<16.4f} {info['p50_query_latency_ms']:<18.3f} {info['p95_query_latency_ms']:<18.3f} {info['top10_agreement_with_flat']*100:<16.1f}%")
print("-" * 90)

### 6. Record Retrieval Examples (1 Success & 1 Honest Failure)

In [ ]:
successful_ex = None
worst_failure = None
for log in test_logs:
    if log.get("recall@5") == 1.0 and successful_ex is None:
        successful_ex = log
    elif log.get("recall@5") == 0.0 and worst_failure is None:
        worst_failure = log

retrieval_examples = {
    "successful_example": successful_ex or test_logs[0],
    "worst_failure_example": worst_failure or test_logs[-1],
}

print("\n--- Successful Retrieval Example ---")
print(f"Query ID: {retrieval_examples['successful_example']['query_id']}")
print(f"Target Pages: {retrieval_examples['successful_example']['target_pages']}")
print(f"Retrieved Pages: {retrieval_examples['successful_example']['retrieved_page_ids'][:5]}")

print("\n--- Worst Retrieval Failure Example ---")
print(f"Query ID: {retrieval_examples['worst_failure_example']['query_id']}")
print(f"Target Pages: {retrieval_examples['worst_failure_example']['target_pages']}")
print(f"Retrieved Pages: {retrieval_examples['worst_failure_example']['retrieved_page_ids'][:5]}")

### 7. Build Production KB & Emit Final Evidence ZIP

In [ ]:
import zipfile

# Build Production FlatIP Index
idx_flat = faiss.IndexFlatIP(final_doc_embs.shape[1])
idx_flat.add(final_doc_embs)
faiss.write_index(idx_flat, str(KB_DIR / "index.faiss"))

with open(KB_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in winning_chunks:
        f.write(json.dumps({
            "chunk_id": c.chunk_id, "doc_id": c.doc_id, "page_id": c.page_id,
            "text": c.text, "word_count": c.word_count, "strategy": c.strategy,
            "parent_id": c.parent_id, "parent_text": c.parent_text,
            "section_title": c.section_title,
        }) + "\n")

index_stats = {
    "index_type": "IndexFlatIP",
    "embedding_model": winning_model_id,
    "embedding_dimension": final_doc_embs.shape[1],
    "total_chunks": len(winning_chunks),
    "avg_words_per_chunk": float(np.mean([c.word_count for c in winning_chunks])),
    "index_size_bytes": (KB_DIR / "index.faiss").stat().st_size,
}

selected_config = f"""# Production Knowledge Base Configuration (Locked in Stage 4)
doc_agent:
  index:
    chunk_strategy: "{winning_strategy}"
    embedding_model: "{winning_model_id}"
    dimension: {final_doc_embs.shape[1]}
    index_type: "IndexFlatIP"
    metric: "INNER_PRODUCT"
"""
(OUT_DIR / "selected-config.yaml").write_text(selected_config, encoding="utf-8")

evidence_summary = f"""# Stage 4 Final Evidence Summary

## 1. Locked Production Stack
- **Embedding Model**: `{winning_model_id}` ({final_doc_embs.shape[1]}-d)
- **Chunking Strategy**: `{winning_strategy}` ({len(winning_chunks)} chunks, avg {index_stats['avg_words_per_chunk']:.1f} words)
- **Vector Search Index**: `IndexFlatIP` ({index_stats['index_size_bytes'] / 1024:.1f} KB)

## 2. Final Untouched Test Performance
- **Single-Page Recall@1**: {final_metrics['single_page_recall@1']:.4f}
- **Single-Page Recall@5**: {final_metrics['single_page_recall@5']:.4f} (95% CI: {final_metrics['recall@5_ci_95']})
- **Single-Page MRR@10**: {final_metrics['single_page_mrr@10']:.4f}
- **Multi-Page Coverage@10**: {final_metrics['multi_page_coverage@10']:.4f}
- **Multi-Page All-Found@10**: {final_metrics['multi_page_all_found@10']:.4f}
"""
(OUT_DIR / "evidence-summary.md").write_text(evidence_summary, encoding="utf-8")
(OUT_DIR / "final-results.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
(OUT_DIR / "faiss-comparison.json").write_text(json.dumps(faiss_summary, indent=2), encoding="utf-8")
(OUT_DIR / "index-statistics.json").write_text(json.dumps(index_stats, indent=2), encoding="utf-8")
(OUT_DIR / "retrieval-examples.json").write_text(json.dumps(retrieval_examples, indent=2), encoding="utf-8")

with open(OUT_DIR / "final-per-query.jsonl", "w", encoding="utf-8") as f:
    for log in test_logs:
        f.write(json.dumps(log) + "\n")

zip_path = OUT_DIR / "stage4-final-evidence.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in [
        "final-results.json",
        "final-per-query.jsonl",
        "faiss-comparison.json",
        "index-statistics.json",
        "retrieval-examples.json",
        "selected-config.yaml",
        "evidence-summary.md",
    ]:
        zf.write(OUT_DIR / fname, arcname=fname)

print(f"\nCreated Final Evidence Package: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")